# Understanding the student dataset

This exploration asks what the dataset contains, which recorded values require
clarification, how student characteristics are distributed, and which numerical
features vary together. Its findings guide the separate preprocessing notebook.

Run from the `notebooks/` directory. The input is
`../data/raw/student_data.csv`. Required packages are pandas, matplotlib, and
seaborn. The environment cell records Python and pandas versions.

Missing responses remain visible throughout the exploration. Replacing missing
failures and absences with zero is an adopted assumption for preprocessing.

This version reorganizes the saved exploration notebook. Every original code
cell is included unchanged. Run all cells from a fresh kernel to generate outputs.


## Establish the dataset

Record the environment, load the source, and inspect dimensions, stored types,
and broad numerical and categorical summaries.

### Environment check

The Python interpreter, version, working directory, and pandas version
are displayed to record the execution environment.

In [ ]:
import sys
import os
import pandas as pd

print(sys.executable)
print(sys.version)
print(os.getcwd())
print(pd.__version__)

### Data loading and initial dimensions

The CSV is loaded into the DataFrame `data`.
The loading options preserve the literal education category "None"
while treating empty fields as missing values.

A table of student variables is created by excluding the exported row index.

In [ ]:
data = pd.read_csv(
    '../data/raw/student_data.csv',
    keep_default_na=False,
    na_values=['']
)

In [ ]:
student_data = data.drop(columns=['Unnamed: 0'])

student_data.shape

### Column types and structure

Inspect each column's stored data type and number of non-missing
values before reviewing variable roles and distributions.

In [ ]:
student_data.info()

### Numerical summary

Summary statistics are inspected to understand the distribution and
range of numerical columns and identify values requiring investigation.

In [ ]:
student_data.describe()

### Categorical summary

Categorical columns are inspected to identify the number of distinct
categories, the most frequent category, and its frequency.

In [ ]:
student_data.describe(include=['str']).T

## Assess completeness and record quality

Check repeated records and missing responses before interpreting individual
features. Investigate the count features whose blanks have an adopted treatment.

### Duplicate records

Records are checked for identical values across the student variables.
The exported row index is excluded because its unique values could
hide otherwise identical records.

In [ ]:
duplicate_count = student_data.duplicated().sum()

duplicate_count

### Missing-value analysis

Missing values are counted for each column to assess data completeness.
The distribution of recorded values is also inspected, including missing
entries, to support later preprocessing decisions.

In [ ]:
missing_counts = student_data.isna().sum()
missing_percentages = (missing_counts / len(student_data) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing count': missing_counts,
    'Missing (%)': missing_percentages
})

missing_summary

### Analysis of failures and absences

The distributions of previous failures and absences are inspected,
including missing entries, to investigate recorded values and the
possible meaning of blanks before deciding on a treatment.

In [ ]:
student_data['failures'].value_counts(dropna=False).sort_index()

In [ ]:
student_data['absences'].value_counts(dropna=False).sort_index()

### Interpretation of missing failures and absences

`failures` contains 551 missing values (84.9%), while `absences`
contains 260 (40.1%). Both columns contain only positive recorded
counts, with no explicit zeros.

This pattern suggests that NaN may represent zero previous failures
or absences. We can adopt this interpretation and replace missing entries
with zero during preprocessing.

## Examine family context

Compare related family fields to identify combinations that require
clarification of their meaning or recording.

### Parental education and occupation

Parental education and occupation are examined together to identify
common combinations and records requiring further investigation.

In [ ]:
mother_education_job = pd.crosstab(
    student_data['Medu'].fillna('Missing'),
    student_data['Mjob'].fillna('Missing')
)

mother_education_job

In [ ]:
father_education_job = pd.crosstab(
    student_data['Fedu'].fillna('Missing'),
    student_data['Fjob'].fillna('Missing')
)

father_education_job

### Parental status and guardian

Parental living arrangements are compared with the recorded guardian.

In [ ]:
parental_status_guardian = pd.crosstab(
    student_data['Pstatus'].fillna('Missing'),
    student_data['guardian'].fillna('Missing')
)

parental_status_guardian

### Family size and parental status

Family-size values are compared with parental living arrangements.

In [ ]:
family_size_parental_status = pd.crosstab(
    student_data['famsize'].astype('string').fillna('Missing'),
    student_data['Pstatus'].fillna('Missing')
)

family_size_parental_status

### Family size and guardian

Family-size values are compared with the recorded guardian.

In [ ]:
family_size_guardian = pd.crosstab(
    student_data['famsize'].astype('string').fillna('Missing'),
    student_data['guardian'].fillna('Missing')
)

family_size_guardian

### Family-size 0 consistency checks

Family size 0 is examined alongside parental status and guardian information
to identify and count combinations requiring further investigation.

In [ ]:
zero_family_together = (
    (student_data['famsize'] == 0) &
    (student_data['Pstatus'] == 'T')
)

zero_family_parent_guardian = (
    (student_data['famsize'] == 0) &
    student_data['guardian'].isin(['mother', 'father'])
)

one_family_together = (
    (student_data['famsize'] == 1) &
    (student_data['Pstatus'] == 'T')
)

family_combination_counts = pd.DataFrame({
    'Combination': [
        'Family size 0 and parents together',
        'Family size 0 and parent as guardian',
        'Family size 1 and parents together'
    ],
    'Records': [
        zero_family_together.sum(),
        zero_family_parent_guardian.sum(),
        one_family_together.sum()
    ]
})

family_combination_counts

### Questionable family-size combinations

There are 41 records with family size zero and parents living together,
53 with family size zero and a parent as guardian, and 31 with family size
one and parents living together. These groups overlap.

If the student is included in family size, zero is inconsistent with a count
of people. A size of one also requires clarification when the student shares
a household with both parents. Parental status and guardianship alone do not
establish household membership, so source clarification is needed.


## Examine travel and study habits

Use residence and school to contextualize travel time, then inspect the
spread of weekly study time, including zero and missing entries.

### Travel time, residence, and school

Travel-time distributions are compared across residence types and
schools. Zero-duration journeys are inspected to identify patterns
requiring clarification.

In [ ]:
travel_summary = student_data.groupby(
    ['school', 'address'],
    dropna=False
)['traveltime'].describe()

travel_summary

### Zero travel times

Zero-duration journeys are counted by school and residence type.

In [ ]:
zero_travel = student_data.loc[
    student_data['traveltime'] == 0,
    ['school', 'address', 'traveltime']
]

zero_travel.groupby(
    ['school', 'address'],
    dropna=False
).size()

### Interpretation of zero travel times

There are 29 recorded zero-minute journeys. Of these, 24 concern urban students
at GP, one concerns an urban student at MS, three concern rural students, and
one has missing residence information.

Very short journeys rounded to zero or differences in recording practices are
possible explanations. The concentration at GP merits clarification, while the
available comparisons cannot establish the cause.


### Study-time distribution

Weekly study time is inspected to identify zero values and unusually
high observations requiring further investigation.

In [ ]:
student_data['studytime'].describe()

### Study-time intervals

Zero and missing study times are counted separately.
Positive study times are grouped into 100-minute intervals
to inspect their distribution.

In [ ]:
study_intervals = pd.cut(
    student_data['studytime'],
    bins=range(0, 1501, 100)
)

study_counts = study_intervals.value_counts().sort_index()

study_summary = pd.DataFrame({
    'Students': study_counts
})

study_summary.loc['Zero study time', 'Students'] = (
    student_data['studytime'].eq(0).sum()
)

study_summary.loc['Missing', 'Students'] = (
    student_data['studytime'].isna().sum()
)

study_summary


## Examine support, activities, and aspirations

Count Yes/No responses and examine whether missing responses tend to
occur for the same students across these features.

### Yes/No variables

Response frequencies are inspected for educational support,
activities, aspirations, internet access, and personal background.
Missing responses are included to assess completeness.

In [ ]:
yes_no_columns = [
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

yes_no_summary = pd.DataFrame(
    index=yes_no_columns,
    columns=['Yes', 'No', 'Missing']
)

for column in yes_no_columns:
    yes_no_summary.loc[column, 'Yes'] = student_data[column].eq('yes').sum()
    yes_no_summary.loc[column, 'No'] = student_data[column].eq('no').sum()
    yes_no_summary.loc[column, 'Missing'] = student_data[column].isna().sum()

yes_no_summary

### Missing-value overlap in Yes/No features

Students with missing responses in each pair of Yes/No features
are counted to examine how often unanswered fields occur together.

In [ ]:
yes_no_columns = [
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

missing_indicators = student_data[yes_no_columns].isna().astype(int)

missing_overlap = missing_indicators.T.dot(missing_indicators)

missing_overlap

### Interpretation of missing-value overlap

Missing responses in the Yes/No features rarely occur together.
Of the 140 students missing at least one response, 135 are missing
exactly one and 5 are missing two or more.

The largest pairwise overlaps involve only two students, between
`schoolsup` and `paid`, and between `famsup` and `nursery`.
This suggests that missing responses are mostly scattered across
students.

## Examine relationships, leisure, and health ratings

Inspect the ordered rating distributions and shared missing responses
before drawing comparisons between these student characteristics.

### Rating-value counts

Responses from 1 to 5 and missing entries are counted for each
rating feature to examine their distributions and completeness.

In [ ]:
rating_columns = [
    'famrel', 'freetime', 'goout',
    'Dalc', 'Walc', 'health'
]

rating_summary = pd.DataFrame(
    index=rating_columns,
    columns=[1, 2, 3, 4, 5, 'Missing']
)

for column in rating_columns:
    for rating in range(1, 6):
        rating_summary.loc[column, rating] = (
            student_data[column].eq(rating).sum()
        )

    rating_summary.loc[column, 'Missing'] = (
        student_data[column].isna().sum()
    )

rating_summary

### Missing-value overlap in rating features

Students with missing responses in each pair of rating features
are counted to examine how often unanswered fields occur together.

In [ ]:
rating_missing_indicators = (
    student_data[rating_columns].isna().astype(int)
)

rating_missing_overlap = (
    rating_missing_indicators.T.dot(rating_missing_indicators)
)

rating_missing_overlap

### Interpretation of missing-value overlap

Missing responses rarely overlap across the six rating features.
The largest pairwise overlap involves three students missing both
`goout` and `Walc`. All other pairs share at most two missing responses.

These counts suggest limited overlap in unanswered ratings.

## Examine academic performance

Review grades across the three periods and follow students with recorded
zeros to understand whether those values persist.

### Grade distributions

The three grade features are examined through summary statistics and
frequency counts. Missing values and recorded zeros are counted separately.

In [ ]:
grade_columns = ['G1', 'G2', 'G3']

display(student_data[grade_columns].describe())

grade_counts = pd.DataFrame({
    column: student_data[column].value_counts()
    for column in grade_columns
}).sort_index()

display(grade_counts)


### Students with zero grades

Students with a zero in any grading period are examined across all three
periods to identify when zeros occur and whether they persist.

In [ ]:
zero_grade_students = student_data.loc[
    student_data[grade_columns].eq(0).any(axis=1),
    ['G1', 'G2', 'G3']
]

display(zero_grade_students)

### Grade analysis conclusion

The student with G1 = 0, G2 = 11, and G3 = 11 shows that a zero in the
first period can be followed by positive grades. However, every student
with G2 = 0 also had G3 = 0.

The reason for the recorded zeros remains unknown. The recorded zero grades are retained
because there is insufficient evidence that they are errors.

### Yes/No response percentages

Percentages show the distribution of Yes, No, and missing responses for each
binary feature, using all students as the denominator.


In [ ]:
yes_no_percentages = pd.DataFrame(index=yes_no_columns)

for column in yes_no_columns:
    yes_no_percentages.loc[column, 'Yes'] = (
        student_data[column].eq('yes').sum() / len(student_data) * 100
    )
    yes_no_percentages.loc[column, 'No'] = (
        student_data[column].eq('no').sum() / len(student_data) * 100
    )
    yes_no_percentages.loc[column, 'Missing'] = (
        student_data[column].isna().sum() / len(student_data) * 100
    )

display(yes_no_percentages.round(2))


### Rating percentages

Percentages show the distribution across ratings 1 to 5 and missing responses.
Each feature uses all students as the denominator.


In [ ]:
rating_percentages = pd.DataFrame(index=rating_columns)

for column in rating_columns:
    for rating in range(1, 6):
        rating_percentages.loc[column, rating] = (
            student_data[column].eq(rating).sum() / len(student_data) * 100
        )
    rating_percentages.loc[column, 'Missing'] = (
        student_data[column].isna().sum() / len(student_data) * 100
    )

display(rating_percentages.round(2))


### Remaining categorical distributions

Each remaining categorical feature has its own horizontal percentage table.
Missing responses are included in the denominator and displayed explicitly.


In [ ]:
categorical_columns = student_data.select_dtypes(
    include=['str', 'object']
).columns

remaining_columns = categorical_columns.drop(yes_no_columns)

for column in remaining_columns:
    remaining_percentages = (
        student_data[column]
        .fillna('Missing')
        .value_counts(normalize=True)
        .mul(100)
        .to_frame(name=column)
        .T
    )
    display(remaining_percentages.round(2))


## Connect the numerical features

After inspecting individual features, examine their linear relationships.
The Pearson matrix uses the available observations for each pair.

### Correlation between numerical features

A Pearson correlation matrix is calculated for the numerical features
to examine the direction and strength of linear relationships.

Each coefficient uses students with recorded values for both features.
The number of observations can therefore differ between comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

correlation_matrix = student_data.corr(numeric_only=True)

plt.figure(figsize=(16, 12))

sns.heatmap(
    data=correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0
)

plt.title('Pearson correlation between numerical features')
plt.tight_layout()
plt.show()

### Interpretation of numerical relationships

The grade features form the clearest correlated group. G1 and G2 have a
correlation of approximately 0.86, G1 and G3 of 0.83, and G2 and G3 of 0.92.
This overlap matters when deciding how much weight academic performance
receives in clustering.

Workday and weekend alcohol consumption correlate at approximately 0.62.
Going out correlates with weekend consumption at 0.40 and free time at 0.35.
These associations suggest related aspects of social behaviour.

Family context remains a thematic grouping. Family size and family relationship
quality correlate at approximately -0.02, and most other family features are
categorical and excluded from this matrix. Ordinal ratings and substantial
missingness in the count features require caution when interpreting Pearson
coefficients. The calculations describe recorded associations.

The heatmap follows Practical 2, Data_Preprocessing.ipynb, cells 139-140.


### Correlation between Yes/No features

Yes and No are encoded as 1 and 0 in a separate analysis copy. Pearson
correlations examine binary associations using available observations per pair.


In [ ]:
yes_no_columns = [
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

yes_no_numeric = student_data[yes_no_columns].copy()

for column in yes_no_columns:
    yes_no_numeric[column] = yes_no_numeric[column].map({'yes': 1, 'no': 0})

yes_no_correlation = yes_no_numeric.corr(method='pearson')

plt.figure(figsize=(10, 8))
sns.heatmap(
    yes_no_correlation,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0
)
plt.title('Pearson correlation between Yes/No features')
plt.tight_layout()
plt.show()


### Correlation between rating features

Spearman correlations examine associations between ordered 1-5 ratings.
Missing responses are excluded separately for each pair. Practical 2 discusses
Spearman in cell 138. This applies that method to the project's rating features.


In [ ]:
rating_columns = ['famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health']

rating_correlation = student_data[rating_columns].corr(method='spearman')

plt.figure(figsize=(9, 7))
sns.heatmap(
    rating_correlation,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0
)
plt.title('Spearman correlation between rating features')
plt.tight_layout()
plt.show()


## Findings guiding preprocessing

### Dataset structure and completeness

The dataset contains 649 students and 33 analytical features after excluding
the exported row index. No exact duplicate records were identified across
these features.

The variables cover demographic characteristics, family background,
educational support, study habits, social behaviour, health, and academic
performance. Students range from 15 to 22 years old.

Age and all three grade variables are complete. Every other feature contains
missing entries, with particularly high missingness in `failures`
(551 students, 84.9%) and `absences` (260 students, 40.1%). Consequently,
preprocessing decisions for these two features will affect a substantial
proportion of the dataset.

### Family information and questionable combinations

Family size presents the clearest unresolved interpretation problem.
There are 57 students with a recorded family size of zero. Among them,
41 have parents recorded as living together and 53 have a mother or father
as their guardian. A further 31 students have family size one and parents
living together. These groups overlap and should be interpreted separately.

If family size includes the student, zero is inconsistent with a count of
family members. Parental status and guardianship provide additional context,
although they do not establish who shares the student's household.
The meaning or recording of `famsize` therefore requires clarification
before selecting a treatment.


### Missing failures and absences

Both `failures` and `absences` contain positive recorded counts and no
explicit zeros. This pattern supports the hypothesis that blanks may
represent zero occurrences.

The adopted preprocessing decision is to replace missing entries in these
two features with zero. This remains an explicit analytical assumption,
as the exploration cannot confirm how the blanks were generated.

### Travel and study time

Travel time has a median of 13 minutes, with recorded values ranging from
0 to 144 minutes. Within both schools, rural students have longer median
journeys than urban students.

There are 29 zero-minute journeys, including 24 among urban students at GP.
This concentration warrants investigation. Very short journeys rounded to
zero and differences in recording practices are possible explanations,
but the available comparisons cannot establish the cause.

Weekly study time has a median of 186 minutes and a mean of approximately
242 minutes. The higher mean and maximum of 1,465 minutes indicate a
distribution extending towards high study times. There are 502 students
reporting positive study times of up to 300 minutes, alongside three
recorded zeros and 14 missing entries.

### Yes/No features

The response counts show that aspirations for higher education, nursery
attendance, and internet access are common. Family educational support is
also frequently reported. School support and paid additional classes are
less common, while participation in extracurricular activities is more
evenly divided.

Missing responses rarely overlap within these eight features. The largest
pairwise overlaps involve two students. This indicates limited shared
missingness across the features examined, although the reason for unanswered
responses remains unknown.

### Rating features

All recorded values in the six rating features fall within the expected
1–5 range. Family relationships are concentrated at ratings 4 and 5,
while free time and going out most frequently receive rating 3.

Workday alcohol consumption is concentrated at rating 1. Weekend consumption
has more responses at higher ratings. Health responses span the full scale,
with rating 5 the most frequent.

Missing-value overlap is also limited across these features. The largest
pairwise overlap involves three students missing both `goout` and `Walc`.
All other pairs share at most two missing responses.

### Academic performance

All three grade features contain 649 observations and range from 0 to 19.
Mean grades increase from approximately 11.40 in G1 to 11.57 in G2 and
11.91 in G3. The medians are 11, 11, and 12 respectively. These summaries
show a modest increase in performance across periods at the dataset level.

There is one zero in G1, seven in G2, and fifteen in G3. The student at
index 0 has G1 = 0, G2 = 11, and G3 = 11, demonstrating that an initial
zero can be followed by positive grades. Every student with G2 = 0 also
has G3 = 0.

The causes of these zeros remain uncertain. They are retained because
the exploration provides insufficient evidence that they are recording
errors.

### Implications for preprocessing and clustering

The exploration identifies differences in feature scales, incomplete
responses, uneven category frequencies, and unresolved values that require
documented preprocessing decisions.

Family-size coding remains an important open issue. The zero-fill treatment
for missing failures and absences must remain documented as an assumption.
Travel-time zeros, high study times, and zero grades require interpretation
consistent with the available evidence.

The findings provide a basis for preparing the data for clustering.
Cluster selection and interpretation will require further analysis of
relationships between features and validation of the resulting groups.